# source
https://silico.biotoul.fr/
p/M2BBS_GDNS-APG
notebook :
https://silico.biotoul.fr/enseignement/m2bbs/gdns-ap/GDNS-APG.Mise.en.oeuvre.html

# IMPORT LIBRARY

In [4]:
#!/usr/bin/env python

import argparse
from os.path import isfile
import json
from scipy.stats import binom, hypergeom,chi2 #pip install scipy

# CREATE PARSER

In [5]:
# SCRIPT PARAMETERS
parser = argparse.ArgumentParser(description='Search enriched terms/categories in the provided (gene) set')
parser.add_argument('-q', '--query', required=True, help='Query set.')
parser.add_argument('-t', '--sets', required=True, help='Target sets filename.')
parser.add_argument('-a', '--alpha', required=False, type=float, default=0.05, help='Significance threshold.')
parser.add_argument('-c', '--adjust', required=False, action="store_true", help='Adjust for multiple testing (FDR).')
parser.add_argument('-m', '--measure', required=False, default='binomial', help='Dissimilarity index: binomial (default), hypergeometric, chi2 or coverage. chi2 and coverage are NOT YET IMPLEMENTED')
parser.add_argument('-l', '--limit', required=False, type=int, default=0, help='Maximum number of results to report.')
parser.add_argument('-v', '--verbose', required=False, action="store_true", help='Talk a lot.')

_StoreTrueAction(option_strings=['-v', '--verbose'], dest='verbose', nargs=0, const=True, default=False, type=None, choices=None, help='Talk a lot.', metavar=None)

## A PARTIR DE KEYWORDS (set) et query (set1)

In [6]:
# param = parser.parse_args(['-q', '/chemin/vers/votre/fichier/query.txt', '-t', '/chemin/vers/votre/fichier/ref.sets/uniprot.keywords.sets.json'])
param = parser.parse_args(['-q','../query/set1.txt','-t','../ref.sets/uniprot.keywords.sets.json','-c','-v','-l','10'])

In [64]:
# LOAD QUERY
text = param.query
query = set()
if isfile(text):
    with open(text) as f:
        content = ' '.join(f.read().split('\n')).split()
        query |= set(content)
else: # parse string
    query |= set(text.split())

if param.verbose:
  print(f'query set: {query}')

query set: {'b2913', 'b0692', 'b3110', 'b2413', 'b1918', 'b3213', 'b4013', 'b2019', 'b2599', 'b3957', 'b0031', 'b1296', 'b1265', 'b2678', 'b3772', 'b2310', 'b0199', 'b3270', 'b2478', 'b3269', 'b4141', 'b0576', 'b2600', 'b2421', 'b0529', 'b0112', 'b3458', 'b3653', 'b0860', 'b1693', 'b1262', 'b2014', 'b0032', 'b1473', 'b3457', 'b0388', 'b1920', 'b0001', 'b0811', 'b3941', 'b0809', 'b3237', 'b2414', 'b2329', 'b3771', 'b2578', 'b2551', 'b2308', 'b4208', 'b4024', 'b2763', 'b3268', 'b4115', 'b3774', 'b2025', 'b0166', 'b1729', 'b3709', 'b3455', 'b3770', 'b2307', 'b3370', 'b0033', 'b2018', 'b4019', 'b2764', 'b0242', 'b0273', 'b0003', 'b0402', 'b0386', 'b3940', 'b4054', 'b0074', 'b0159', 'b0654', 'b3829', 'b3116', 'b1704', 'b3670', 'b3271', 'b3939', 'b1605', 'b3212', 'b1015', 'b2131', 'b0071', 'b3281', 'b4243', 'b2129', 'b2677', 'b4254', 'b3672', 'b3389', 'b2156', 'b4488', 'b2923', 'b3161', 'b0863', 'b3671', 'b2365', 'b2663', 'b0077', 'b0754', 'b0861', 'b4132', 'b1907', 'b2023', 'b1798', 'b1917'

In [8]:
# LOAD REFERENCE SETS
sets = json.loads(open(param.sets).read())
# set est une liste de dictionnaires
if param.verbose:
    print('first target sets: ', sets[0:2])

first target sets:  [{'id': '2Fe-2S', 'desc': 26, 'elements': ['b0286', 'b0724', 'b0775', 'b0872', 'b0873', 'b1392', 'b1654', 'b1684', 'b1802', 'b1803', 'b2236', 'b2283', 'b2285', 'b2525', 'b2530', 'b2531', 'b2538', 'b2540', 'b2868', 'b2881', 'b3337', 'b3771', 'b4063', 'b4153', 'b4178', 'b4367']}, {'id': '3D-structure', 'desc': 1665, 'elements': ['b0002', 'b0004', 'b0006', 'b0008', 'b0009', 'b0010', 'b0014', 'b0015', 'b0019', 'b0023', 'b0028', 'b0029', 'b0031', 'b0032', 'b0033', 'b0038', 'b0040', 'b0046', 'b0047', 'b0048', 'b0051', 'b0052', 'b0053', 'b0054', 'b0058', 'b0059', 'b0060', 'b0061', 'b0062', 'b0064', 'b0068', 'b0073', 'b0076', 'b0078', 'b0080', 'b0082', 'b0084', 'b0085', 'b0086', 'b0088', 'b0090', 'b0091', 'b0092', 'b0093', 'b0094', 'b0095', 'b0096', 'b0097', 'b0098', 'b0099', 'b0101', 'b0102', 'b0103', 'b0113', 'b0114', 'b0115', 'b0116', 'b0118', 'b0121', 'b0123', 'b0125', 'b0126', 'b0131', 'b0133', 'b0134', 'b0140', 'b0142', 'b0144', 'b0145', 'b0146', 'b0147', 'b0148', 'b0

In [9]:
sets

[{'id': '2Fe-2S',
  'desc': 26,
  'elements': ['b0286',
   'b0724',
   'b0775',
   'b0872',
   'b0873',
   'b1392',
   'b1654',
   'b1684',
   'b1802',
   'b1803',
   'b2236',
   'b2283',
   'b2285',
   'b2525',
   'b2530',
   'b2531',
   'b2538',
   'b2540',
   'b2868',
   'b2881',
   'b3337',
   'b3771',
   'b4063',
   'b4153',
   'b4178',
   'b4367']},
 {'id': '3D-structure',
  'desc': 1665,
  'elements': ['b0002',
   'b0004',
   'b0006',
   'b0008',
   'b0009',
   'b0010',
   'b0014',
   'b0015',
   'b0019',
   'b0023',
   'b0028',
   'b0029',
   'b0031',
   'b0032',
   'b0033',
   'b0038',
   'b0040',
   'b0046',
   'b0047',
   'b0048',
   'b0051',
   'b0052',
   'b0053',
   'b0054',
   'b0058',
   'b0059',
   'b0060',
   'b0061',
   'b0062',
   'b0064',
   'b0068',
   'b0073',
   'b0076',
   'b0078',
   'b0080',
   'b0082',
   'b0084',
   'b0085',
   'b0086',
   'b0088',
   'b0090',
   'b0091',
   'b0092',
   'b0093',
   'b0094',
   'b0095',
   'b0096',
   'b0097',
   'b0098',
  

## 1.1.2.4 Test statistique

### 1.1.2.4.1 Taille de la population

In [10]:
len_query = len(query)
len_sets = len(sets) # nombre d ensembles de reference
len_query, len_sets 

(181, 334)

In [11]:
sets[0].keys()

dict_keys(['id', 'desc', 'elements'])

In [12]:
# j accede a la liste elements dans le premier dictionnaire de sets
sets[0]['elements']

['b0286',
 'b0724',
 'b0775',
 'b0872',
 'b0873',
 'b1392',
 'b1654',
 'b1684',
 'b1802',
 'b1803',
 'b2236',
 'b2283',
 'b2285',
 'b2525',
 'b2530',
 'b2531',
 'b2538',
 'b2540',
 'b2868',
 'b2881',
 'b3337',
 'b3771',
 'b4063',
 'b4153',
 'b4178',
 'b4367']

In [13]:
# et pour avoir la longeuer de cette liste :
len(sets[0]['elements']), len(sets[1]['elements']) #len de sets = 

(26, 1665)

In [14]:
# COMPUTE POPULATION SIZE, deduire la taille de la population dans ref set
population = set() # faire en sorte que population soit un ensemble
for s in sets:
  valeur_cle3 = s.get('elements')
  population.update(valeur_cle3)

# Maintenant, "valeurs_uniques" contient toutes les valeurs uniques de la troisième clé ('cle3')
n_gene_total = len(population)

# Affichez le nombre de valeurs uniques d elements dans la population
print(n_gene_total) # 4400

4400


### 1.1.2.4.2 Comparaison des ensembles cibles à l’ensemble requête

In [15]:
len(sets[0]['elements'])

26

In [16]:
p = len(sets[5]['elements']) / n_gene_total #probabilite de succes a chaque essai
p

0.00045454545454545455

In [17]:
# EVALUATE SETS
results = []
query_size = len(query)
for s in sets:
    elements = set(s['elements']) # liste d elements unique dans elements
    common_elements = elements.intersection(query) #k, nombre de succes dans la population = commun entre query et elements
    k = len(common_elements) # nombre de succes dans la population = commun entre query et elements
    if param.measure=='binomial': # par défaut binom.cdf(<=success, attempts, proba). il nous faut calculer p(X>=x)
        p = 1 - binom.cdf(k - 1, len(elements), query_size / n_gene_total)
    else:
        print(f'sorry, {param.measure} not (yet) implemented')
        exit(1)
    r = { 'id': s['id'], 
         'desc': s['desc'], 
         'common.n':len(common_elements), 
         'target.n': len(elements), 
         'p-value': p, 
         'elements.target': elements, 
         'elements.common': common_elements }
    results.append( r )

if param.verbose:
  print(results)

[{'id': '2Fe-2S', 'desc': 26, 'common.n': 1, 'target.n': 26, 'p-value': 0.6645111651743494, 'elements.target': {'b4367', 'b4153', 'b2868', 'b0872', 'b1392', 'b2530', 'b2538', 'b2236', 'b1803', 'b2881', 'b0775', 'b1684', 'b2531', 'b1654', 'b3337', 'b2525', 'b0873', 'b0724', 'b2283', 'b4178', 'b2285', 'b0286', 'b2540', 'b4063', 'b3771', 'b1802'}, 'elements.common': {'b3771'}}, {'id': '3D-structure', 'desc': 1665, 'common.n': 70, 'target.n': 1665, 'p-value': 0.44316705921606314, 'elements.target': {'b2287', 'b0407', 'b1211', 'b2411', 'b4101', 'b3320', 'b3772', 'b1652', 'b2918', 'b2373', 'b2894', 'b1745', 'b2562', 'b4035', 'b4025', 'b4375', 'b2288', 'b4090', 'b0758', 'b1159', 'b0345', 'b2441', 'b1987', 'b3869', 'b0778', 'b0881', 'b2371', 'b1045', 'b3620', 'b0565', 'b0480', 'b1625', 'b4501', 'b3400', 'b4034', 'b2723', 'b3190', 'b2385', 'b0897', 'b1635', 'b3786', 'b2595', 'b4311', 'b0123', 'b2905', 'b1100', 'b1636', 'b0096', 'b3651', 'b3619', 'b2942', 'b1390', 'b2910', 'b3707', 'b0907', 'b25

In [18]:
# results est une liste de dictionnaires
len(results)

334

In [19]:
# afficher toutes les valeurs de la clé 'p-value' dans results
for r in results:
    print(r['p-value'])

0.6645111651743494
0.44316705921606314
0.25475693679208333
0.8395215401495404
1.0
1.0
1.0
0.028420287029181313
0.9207224726048383
0.8603238039509017
0.30646348011589
2.3208778879846292e-05
1.0
1.0
0.0
0.0
1.0
1.0
0.0010865052730085756
1.0
1.0
1.0
8.54079266311425e-05
1.0
1.0
0.0
1.0
0.0
1.0
1.0
1.0
0.001692200413223155
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.0
0.31480901546376805
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.0002788116390309847
0.004372516544615013
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.154667832410053
1.0
0.05442806797935795
1.0
1.0
1.0
1.0
1.0
1.0
1.9933399375560157e-10
1.0
1.0
0.0010092758963732296
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.9883526806585471
1.0
1.0
1.0
0.999962906148766
1.0
1.0
0.21321029302245242
1.0
0.22278514384573755
1.1779571484904494e-07
1.0
1.0
1.431494575099279e-07
0.41831787797035813
0.982272091806977
1.0
1.0
1.0
1.0
1.0
1.0
0.02493264943229978
0.2485354136688378
1.0
1.0
1.0
1.0
1.0
0.1422376550392841
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.001692200

### 1.1.2.5 Affichage des résultats significatifs

In [20]:
significant_results = []
# count_significant = 0
# PRINT SIGNIFICANT RESULTS
results.sort(key=lambda an_item: an_item['p-value'])
for r in results:
    if r['p-value'] <= param.alpha: 
      significant_results.append((r['p-value']))
      #count_significant += 1
      #break
    # OUTPUT
    print("{}\t{}\t{}/{}\t{}\t{}".format( r['id'], r['p-value'], r['common.n'], r['target.n'], r['desc'], ', '.join(r['elements.common'])))

Amino-acid biosynthesis	0.0	102/104	104	b2913, b2413, b4013, b3213, b2019, b2599, b3957, b0031, b1265, b3772, b2478, b2600, b2421, b0529, b1693, b1262, b0032, b0388, b0001, b3941, b2414, b3237, b2329, b3771, b2551, b4024, b2763, b3774, b2025, b0166, b3770, b0033, b2018, b4019, b2764, b0242, b0273, b0003, b0386, b3940, b4054, b0074, b0159, b3829, b1704, b3670, b3939, b3212, b0071, b3281, b4243, b4254, b3672, b3389, b4488, b3671, b0077, b0754, b2023, b1264, b3172, b0907, b3958, b0002, b3359, b0908, b3390, b2472, b2022, b2024, b3828, b1261, b2601, b3008, b3769, b2838, b1622, b4388, b3766, b3773, b2021, b3809, b0674, b3433, b1263, b3959, b3960, b0072, b3607, b2020, b0078, b0261, b0243, b3744, b0073, b1260, b2026, b2818, b3938, b0004, b1275, b0075
Amino-acid transport	0.0	79/82	82	b0692, b3110, b1918, b1296, b2678, b2310, b0199, b3270, b3269, b4141, b0576, b0112, b3653, b0860, b2014, b1473, b3457, b1920, b0811, b0809, b2578, b2308, b4208, b3268, b4115, b1729, b3709, b3455, b2307, b3370, b04

Exemple d’utilisation avec set.01.txt :
./scripts/blastsets.json.py -q query.sets/set.01.txt -t reference.sets/uniprot.keywords.sets.json  | column -ts $'\t'
 ### POur avoir le nombre de res significatif
./scripts/blastsets.json.py --adjust -q query.sets/set.01.txt -t reference.sets/uniprot.keywords.sets.json | wc -l

In [21]:
print(f"Nombre de résultats significatifs : {len(significant_results)}") # dois avoir 41

Nombre de résultats significatifs : 41


#### FDR avec p valeur ajustée
P+m / i et si p_adjusted > 1 alpha = 1

In [22]:
results[0]['p-value']

0.0

In [23]:
significant_results = []
#count_significant = 0
# PRINT SIGNIFICANT RESULTS
results.sort(key=lambda an_item: an_item['p-value'])
for r in results:
  if r['p-value'] < param.alpha: 
    significant_results.append((r['p-value']))
    print("{}\t{}\t{}/{}\t{}\t{}".format( r['id'], r['p-value'], r['common.n'], r['target.n'], r['desc'], ', '.join(r['elements.common'])))
    print(f"Number of significant results: {len(significant_results)}")

  if param.adjust:
    p_ajusted = []
    for i, proba in enumerate(significant_results):
      p = (proba*len_sets) / (i+1)
      if p <= param.alpha :
        p_ajusted.append(p) 

Amino-acid biosynthesis	0.0	102/104	104	b2913, b2413, b4013, b3213, b2019, b2599, b3957, b0031, b1265, b3772, b2478, b2600, b2421, b0529, b1693, b1262, b0032, b0388, b0001, b3941, b2414, b3237, b2329, b3771, b2551, b4024, b2763, b3774, b2025, b0166, b3770, b0033, b2018, b4019, b2764, b0242, b0273, b0003, b0386, b3940, b4054, b0074, b0159, b3829, b1704, b3670, b3939, b3212, b0071, b3281, b4243, b4254, b3672, b3389, b4488, b3671, b0077, b0754, b2023, b1264, b3172, b0907, b3958, b0002, b3359, b0908, b3390, b2472, b2022, b2024, b3828, b1261, b2601, b3008, b3769, b2838, b1622, b4388, b3766, b3773, b2021, b3809, b0674, b3433, b1263, b3959, b3960, b0072, b3607, b2020, b0078, b0261, b0243, b3744, b0073, b1260, b2026, b2818, b3938, b0004, b1275, b0075
Number of significant results: 1
Amino-acid transport	0.0	79/82	82	b0692, b3110, b1918, b1296, b2678, b2310, b0199, b3270, b3269, b4141, b0576, b0112, b3653, b0860, b2014, b1473, b3457, b1920, b0811, b0809, b2578, b2308, b4208, b3268, b4115, b1729

In [24]:
len(p_ajusted) # 35 ok !!!

35

# 1.1.3 Autres ensembles d’ensembles : Interpro
les domaines Interpro à partir du protéome téléchargé UniProt Proteome (colonne InterPro) -> faire avec R, export en json

## 1.1.2.4 Test statistique

### 1.1.2.4.1 Taille de la population

In [25]:
# param = parser.parse_args(['-q', '/chemin/vers/votre/fichier/query.txt', '-t', '/chemin/vers/votre/fichier/ref.sets/uniprot.keywords.sets.json'])
param = parser.parse_args(['-q','../query/set1.txt',
                           '-t','../ref.sets/uniprot.interpro.sets.json',
                           '-c','-v','-l','10'])

# LOAD QUERY
text = param.query
query = set()
if isfile(text):
    with open(text) as f:
        content = ' '.join(f.read().split('\n')).split()
        query |= set(content)
else: # parse string
    query |= set(text.split())

if param.verbose:
  print(f'query set: {query}')

# LOAD REFERENCE SETS
sets = json.loads(open(param.sets).read())# set est une liste de dictionnaires
if param.verbose:
    print('first target sets: ', sets[0:2])

query set: {'b2913', 'b0692', 'b3110', 'b2413', 'b1918', 'b3213', 'b4013', 'b2019', 'b2599', 'b3957', 'b0031', 'b1296', 'b1265', 'b2678', 'b3772', 'b2310', 'b0199', 'b3270', 'b2478', 'b3269', 'b4141', 'b0576', 'b2600', 'b2421', 'b0529', 'b0112', 'b3458', 'b3653', 'b0860', 'b1693', 'b1262', 'b2014', 'b0032', 'b1473', 'b3457', 'b0388', 'b1920', 'b0001', 'b0811', 'b3941', 'b0809', 'b3237', 'b2414', 'b2329', 'b3771', 'b2578', 'b2551', 'b2308', 'b4208', 'b4024', 'b2763', 'b3268', 'b4115', 'b3774', 'b2025', 'b0166', 'b1729', 'b3709', 'b3455', 'b3770', 'b2307', 'b3370', 'b0033', 'b2018', 'b4019', 'b2764', 'b0242', 'b0273', 'b0003', 'b0402', 'b0386', 'b3940', 'b4054', 'b0074', 'b0159', 'b0654', 'b3829', 'b3116', 'b1704', 'b3670', 'b3271', 'b3939', 'b1605', 'b3212', 'b1015', 'b2131', 'b0071', 'b3281', 'b4243', 'b2129', 'b2677', 'b4254', 'b3672', 'b3389', 'b2156', 'b4488', 'b2923', 'b3161', 'b0863', 'b3671', 'b2365', 'b2663', 'b0077', 'b0754', 'b0861', 'b4132', 'b1907', 'b2023', 'b1798', 'b1917'

In [26]:
len_query = len(query)
len_sets = len(sets) # nombre d ensembles de reference
len_query, len_sets

(181, 2083)

In [27]:
sets[0].keys()

dict_keys(['id', 'desc', 'elements'])

In [28]:
# j accede a la liste elements dans le premier dictionnaire de sets
sets[0]['elements']

['b0400',
 'b0619',
 'b1201',
 'b1285',
 'b1323',
 'b1341',
 'b1489',
 'b2067',
 'b2219',
 'b2869',
 'b3072',
 'b3210',
 'b3869',
 'b4125']

In [29]:
# et pour avoir la longeuer de cette liste :
len(sets[0]['elements']), len(sets[1]['elements']) #len de sets = 

(14, 11)

In [30]:
# COMPUTE POPULATION SIZE, deduire la taille de la population dans ref set
population = set() # faire en sorte que population soit un ensemble
for s in sets:
  valeur_cle3 = s.get('elements')
  population.update(valeur_cle3)

# Maintenant, "valeurs_uniques" contient toutes les valeurs uniques de la troisième clé ('cle3')
n_gene_total = len(population)

# Affichez le nombre de valeurs uniques d elements dans la population
print(n_gene_total) # 4400, le meme qu avec keywords

3335


### 1.1.2.4.2 Comparaison des ensembles cibles à l’ensemble requête

In [31]:
len(sets[0]['elements'])

14

In [32]:
p = len(sets[5]['elements']) / n_gene_total #probabilite de succes a chaque essai
p

0.0005997001499250374

In [33]:
sets

[{'id': 'IPR000014',
  'desc': 14,
  'elements': ['b0400',
   'b0619',
   'b1201',
   'b1285',
   'b1323',
   'b1341',
   'b1489',
   'b2067',
   'b2219',
   'b2869',
   'b3072',
   'b3210',
   'b3869',
   'b4125']},
 {'id': 'IPR000015',
  'desc': 11,
  'elements': ['b0139',
   'b0532',
   'b0718',
   'b0940',
   'b1505',
   'b2109',
   'b3046',
   'b3144',
   'b3216',
   'b4317',
   'b4661']},
 {'id': 'IPR000021',
  'desc': 6,
  'elements': ['b0018', 'b1562', 'b4412', 'b4415', 'b4428', 'b4455']},
 {'id': 'IPR000032',
  'desc': 6,
  'elements': ['b1198', 'b2169', 'b2383', 'b2415', 'b3206', 'b3947']},
 {'id': 'IPR000045', 'desc': 2, 'elements': ['b2972', 'b3335']},
 {'id': 'IPR000049', 'desc': 2, 'elements': ['b0041', 'b1697']},
 {'id': 'IPR000056', 'desc': 3, 'elements': ['b3386', 'b4085', 'b4301']},
 {'id': 'IPR000060', 'desc': 3, 'elements': ['b0040', 'b0314', 'b1801']},
 {'id': 'IPR000064',
  'desc': 4,
  'elements': ['b0227', 'b1655', 'b1708', 'b2175']},
 {'id': 'IPR000073',
  'des

In [34]:
# EVALUATE SETS
results = []
query_size = len(query)
for s in sets:
    elements = set(s['elements']) # liste d elements unique dans elements
    common_elements = elements.intersection(query) #k, nombre de succes dans la population = commun entre query et elements
    k = len(common_elements) # nombre de succes dans la population = commun entre query et elements
    if param.measure=='binomial': # par défaut binom.cdf(<=success, attempts, proba). il nous faut calculer p(X>=x)
        p = 1 - binom.cdf(k - 1, len(elements), query_size / n_gene_total)
    else:
        print(f'sorry, {param.measure} not (yet) implemented')
        exit(1)
    r = { 'id': s['id'], 
         'desc': s['desc'], 
         'common.n':len(common_elements), 
         'target.n': len(elements), 
         'p-value': p, 
         'elements.target': elements, 
         'elements.common': common_elements }
    results.append( r )

if param.verbose:
  print(results)

[{'id': 'IPR000014', 'desc': 14, 'common.n': 0, 'target.n': 14, 'p-value': 1.0, 'elements.target': {'b3869', 'b2869', 'b2067', 'b1323', 'b3072', 'b0400', 'b1341', 'b0619', 'b4125', 'b1201', 'b3210', 'b2219', 'b1285', 'b1489'}, 'elements.common': set()}, {'id': 'IPR000015', 'desc': 11, 'common.n': 0, 'target.n': 11, 'p-value': 1.0, 'elements.target': {'b3046', 'b3216', 'b0139', 'b0940', 'b4317', 'b0532', 'b2109', 'b1505', 'b4661', 'b0718', 'b3144'}, 'elements.common': set()}, {'id': 'IPR000021', 'desc': 6, 'common.n': 0, 'target.n': 6, 'p-value': 1.0, 'elements.target': {'b1562', 'b4412', 'b4455', 'b4428', 'b0018', 'b4415'}, 'elements.common': set()}, {'id': 'IPR000032', 'desc': 6, 'common.n': 0, 'target.n': 6, 'p-value': 1.0, 'elements.target': {'b2415', 'b2169', 'b2383', 'b3947', 'b1198', 'b3206'}, 'elements.common': set()}, {'id': 'IPR000045', 'desc': 2, 'common.n': 0, 'target.n': 2, 'p-value': 1.0, 'elements.target': {'b3335', 'b2972'}, 'elements.common': set()}, {'id': 'IPR000049',

In [35]:
# results est une liste de dictionnaires
len(results)

2083

In [36]:
# afficher toutes les valeurs de la clé 'p-value' dans results
for r in results:
    print(r['p-value'])

1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.7379522604618014
0.10560018341653754
1.0
1.0
0.2000489680954688
1.0
1.0
1.0
1.0
1.0
0.0029455437198941947
1.0
1.0
1.0
1.0
1.0
1.0
0.10560018341653754
1.0
1.0
1.0
1.0
1.0
0.2000489680954688
1.0
1.0
1.0
0.008516904974794515
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.10560018341653754
4.766139211076492e-07
1.0
1.0
1.0
0.02638643523223627
1.0
1.0
1.0
0.15414182263740916
1.0
0.2000489680954688
1.0
1.0
1.0
0.018981775799489453
0.0029455437198941947
1.0
0.01642038626323028
1.0
0.39480872714665005
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.0029455437198941947
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.10560018341653754
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.2434646013112769
0.4588469230968808
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.10560018341653754
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.2000489680954688
1.0
1.0
1.0
4.437034309856536e-06
1.0
1.0
1.0
1.0
0.10560018341653754
1.0
1.0
1.0
1.0
0.002823527849743712
1.0
1.0
1.0
1.0
1.0
1.0
1.

### 1.1.2.5 Affichage des résultats significatifs

In [37]:
significant_results = []
#count_significant = 0
# PRINT SIGNIFICANT RESULTS
results.sort(key=lambda an_item: an_item['p-value'])
for r in results:
    if r['p-value'] <= param.alpha: 
      significant_results.append((r['p-value']))
      #count_significant += 1
      #break
      # OUTPUT
      print("{}\t{}\t{}/{}\t{}\t{}".format( r['id'], r['p-value'], r['common.n'], r['target.n'], r['desc'], ', '.join(r['elements.common'])))

IPR010065	2.2171153801764376e-13	10/10	10	b2307, b2308, b3270, b1918, b0861, b0654, b3269, b0653, b0862, b0810
IPR004841	3.8859360174114954e-11	9/10	10	b0112, b4208, b2663, b3795, b0402, b0576, b1296, b1453, b2156
IPR004840	6.44808428873489e-10	8/9	9	b0112, b4208, b2663, b3795, b0402, b0576, b1453, b2156
IPR002293	7.734909379664145e-10	9/12	12	b1605, b3370, b0692, b0486, b4115, b4132, b2014, b4141, b1492
IPR018313	1.3870086190692632e-09	7/7	7	b0863, b0860, b3268, b1920, b0811, b2309, b2310
IPR043429	1.3870086190692632e-09	7/7	7	b3270, b1918, b3269, b0861, b0654, b0653, b0810
IPR030679	2.5556208371746436e-08	6/6	6	b2306, b0652, b0809, b0864, b3271, b1917
IPR001638	3.058133013489339e-08	8/12	12	b0863, b0860, b3268, b1920, b0811, b0655, b2309, b2310
IPR045865	3.293623074274166e-08	9/16	16	b0199, b4024, b2913, b0078, b3769, b2599, b3670, b3772, b0002
IPR000515	4.766139211076492e-07	14/52	52	b2307, b2128, b2308, b3270, b0862, b1918, b0861, b3269, b0654, b2130, b0653, b0198, b2678, b0810
IPR

Exemple d’utilisation avec set.01.txt :
./scripts/blastsets.json.py -q query.sets/set.01.txt -t reference.sets/uniprot.keywords.sets.json  | column -ts $'\t'
 ### POur avoir le nombre de res significatif
./scripts/blastsets.json.py --adjust -q query.sets/set.01.txt -t reference.sets/uniprot.keywords.sets.json | wc -l

In [38]:
print(f"Nombre de résultats significatifs : {len(significant_results)}") # 85, expected ?

Nombre de résultats significatifs : 85


#### FDR avec p valeur ajustée
P+m / i et si p_adjusterd > 1 alpha = 1

In [39]:
results[0]['p-value']

2.2171153801764376e-13

In [40]:
significant_results = []
#count_significant = 0
# PRINT SIGNIFICANT RESULTS
results.sort(key=lambda an_item: an_item['p-value'])
for r in results:
  if r['p-value'] < param.alpha: 
    significant_results.append((r['p-value']))
    print("{}\t{}\t{}/{}\t{}\t{}".format( r['id'], r['p-value'], r['common.n'], r['target.n'], r['desc'], ', '.join(r['elements.common'])))

  if param.adjust:
    p_ajusted = []
    for i, proba in enumerate(significant_results):
      p = (proba*len_sets) / (i+1)
      if p <= param.alpha :
        p_ajusted.append(p) 

IPR010065	2.2171153801764376e-13	10/10	10	b2307, b2308, b3270, b1918, b0861, b0654, b3269, b0653, b0862, b0810
IPR004841	3.8859360174114954e-11	9/10	10	b0112, b4208, b2663, b3795, b0402, b0576, b1296, b1453, b2156
IPR004840	6.44808428873489e-10	8/9	9	b0112, b4208, b2663, b3795, b0402, b0576, b1453, b2156
IPR002293	7.734909379664145e-10	9/12	12	b1605, b3370, b0692, b0486, b4115, b4132, b2014, b4141, b1492
IPR018313	1.3870086190692632e-09	7/7	7	b0863, b0860, b3268, b1920, b0811, b2309, b2310
IPR043429	1.3870086190692632e-09	7/7	7	b3270, b1918, b3269, b0861, b0654, b0653, b0810
IPR030679	2.5556208371746436e-08	6/6	6	b2306, b0652, b0809, b0864, b3271, b1917
IPR001638	3.058133013489339e-08	8/12	12	b0863, b0860, b3268, b1920, b0811, b0655, b2309, b2310
IPR045865	3.293623074274166e-08	9/16	16	b0199, b4024, b2913, b0078, b3769, b2599, b3670, b3772, b0002
IPR000515	4.766139211076492e-07	14/52	52	b2307, b2128, b2308, b3270, b0862, b1918, b0861, b3269, b0654, b2130, b0653, b0198, b2678, b0810
IPR

In [41]:
len(p_ajusted) # 30, expected ?

30

# 1.1.3 les GOTerms référencés dans le même fichier d’UniProt (colonne Gene ontology IDs)

## 1.1.2.4 Test statistique
### IMport avec parser

In [42]:
# param = parser.parse_args(['-q', '/chemin/vers/votre/fichier/query.txt', '-t', '/chemin/vers/votre/fichier/ref.sets/uniprot.keywords.sets.json'])
param = parser.parse_args(['-q','../query/set1.txt','-t','../ref.sets/uniprot.go.sets.json','-c','-v','-l','10'])

# LOAD QUERY
text = param.query
query = set()
if isfile(text):
    with open(text) as f:
        content = ' '.join(f.read().split('\n')).split()
        query |= set(content)
else: # parse string
    query |= set(text.split())

if param.verbose:
  print(f'query set: {query}')

# LOAD REFERENCE SETS
sets = json.loads(open(param.sets).read()) # set est une liste de dictionnaires
if param.verbose:
    print('first target sets: ', sets[0:2])

query set: {'b2913', 'b0692', 'b3110', 'b2413', 'b1918', 'b3213', 'b4013', 'b2019', 'b2599', 'b3957', 'b0031', 'b1296', 'b1265', 'b2678', 'b3772', 'b2310', 'b0199', 'b3270', 'b2478', 'b3269', 'b4141', 'b0576', 'b2600', 'b2421', 'b0529', 'b0112', 'b3458', 'b3653', 'b0860', 'b1693', 'b1262', 'b2014', 'b0032', 'b1473', 'b3457', 'b0388', 'b1920', 'b0001', 'b0811', 'b3941', 'b0809', 'b3237', 'b2414', 'b2329', 'b3771', 'b2578', 'b2551', 'b2308', 'b4208', 'b4024', 'b2763', 'b3268', 'b4115', 'b3774', 'b2025', 'b0166', 'b1729', 'b3709', 'b3455', 'b3770', 'b2307', 'b3370', 'b0033', 'b2018', 'b4019', 'b2764', 'b0242', 'b0273', 'b0003', 'b0402', 'b0386', 'b3940', 'b4054', 'b0074', 'b0159', 'b0654', 'b3829', 'b3116', 'b1704', 'b3670', 'b3271', 'b3939', 'b1605', 'b3212', 'b1015', 'b2131', 'b0071', 'b3281', 'b4243', 'b2129', 'b2677', 'b4254', 'b3672', 'b3389', 'b2156', 'b4488', 'b2923', 'b3161', 'b0863', 'b3671', 'b2365', 'b2663', 'b0077', 'b0754', 'b0861', 'b4132', 'b1907', 'b2023', 'b1798', 'b1917'

In [43]:
sets

[{'id': 'GO:0000014',
  'desc': 4,
  'elements': ['b0228', 'b0397', 'b0398', 'b2761']},
 {'id': 'GO:0000018',
  'desc': 4,
  'elements': ['b0393', 'b2733', 'b2831', 'b4170']},
 {'id': 'GO:0000027',
  'desc': 31,
  'elements': ['b0797',
   'b1089',
   'b1343',
   'b1716',
   'b2185',
   'b2511',
   'b2576',
   'b2594',
   'b2606',
   'b3162',
   'b3179',
   'b3180',
   'b3183',
   'b3185',
   'b3294',
   'b3302',
   'b3305',
   'b3308',
   'b3309',
   'b3312',
   'b3313',
   'b3317',
   'b3318',
   'b3320',
   'b3636',
   'b3637',
   'b3780',
   'b3871',
   'b3983',
   'b3984',
   'b4109']},
 {'id': 'GO:0000028',
  'desc': 20,
  'elements': ['b0023',
   'b0051',
   'b0169',
   'b0911',
   'b2566',
   'b2608',
   'b2609',
   'b3065',
   'b3165',
   'b3170',
   'b3180',
   'b3296',
   'b3303',
   'b3306',
   'b3307',
   'b3311',
   'b3314',
   'b3316',
   'b3341',
   'b4161']},
 {'id': 'GO:0000049',
  'desc': 50,
  'elements': ['b0026',
   'b0195',
   'b0423',
   'b0426',
   'b1084',
   '

### 1.1.2.4.1 Taille de la population

In [44]:
len_query = len(query)
len_sets = len(sets) # nombre d ensembles de reference
len_query, len_sets

(181, 2095)

In [45]:
sets[0].keys()

dict_keys(['id', 'desc', 'elements'])

In [46]:
# j accede a la liste elements dans le premier dictionnaire de sets
sets[0]['elements']

['b0228', 'b0397', 'b0398', 'b2761']

In [47]:
# et pour avoir la longeuer de cette liste :
len(sets[0]['elements']), len(sets[1]['elements']) #len de sets = 

(4, 4)

In [48]:
# COMPUTE POPULATION SIZE, deduire la taille de la population dans ref set
population = set() # faire en sorte que population soit un ensemble
for s in sets:
  valeur_cle3 = s.get('elements')
  population.update(valeur_cle3)

# Maintenant, "valeurs_uniques" contient toutes les valeurs uniques de la troisième clé ('cle3')
n_gene_total = len(population)

# Affichez le nombre de valeurs uniques d elements dans la population
print(n_gene_total) # 4400

3944


### 1.1.2.4.2 Comparaison des ensembles cibles à l’ensemble requête

In [49]:
len(sets[0]['elements'])

4

In [50]:
p = len(sets[5]['elements']) / n_gene_total #probabilite de succes a chaque essai
p

0.0005070993914807302

In [51]:
# EVALUATE SETS
results = []
query_size = len(query)
for s in sets:
    elements = set(s['elements']) # liste d elements unique dans elements
    common_elements = elements.intersection(query) #k, nombre de succes dans la population = commun entre query et elements
    k = len(common_elements) # nombre de succes dans la population = commun entre query et elements
    if param.measure=='binomial': # par défaut binom.cdf(<=success, attempts, proba). il nous faut calculer p(X>=x)
        p = 1 - binom.cdf(k - 1, len(elements), query_size / n_gene_total)
    else:
        print(f'sorry, {param.measure} not (yet) implemented')
        exit(1)
    r = { 'id': s['id'], 
         'desc': s['desc'], 
         'common.n':len(common_elements), 
         'target.n': len(elements), 
         'p-value': p, 
         'elements.target': elements, 
         'elements.common': common_elements }
    results.append( r )

if param.verbose:
  print(results)

[{'id': 'GO:0000014', 'desc': 4, 'common.n': 0, 'target.n': 4, 'p-value': 1.0, 'elements.target': {'b0228', 'b0398', 'b0397', 'b2761'}, 'elements.common': set()}, {'id': 'GO:0000018', 'desc': 4, 'common.n': 0, 'target.n': 4, 'p-value': 1.0, 'elements.target': {'b2733', 'b4170', 'b0393', 'b2831'}, 'elements.common': set()}, {'id': 'GO:0000027', 'desc': 31, 'common.n': 0, 'target.n': 31, 'p-value': 1.0, 'elements.target': {'b2594', 'b2511', 'b3183', 'b3309', 'b3636', 'b3637', 'b3180', 'b1716', 'b3320', 'b3984', 'b4109', 'b2606', 'b2576', 'b3317', 'b3302', 'b3312', 'b3983', 'b3313', 'b2185', 'b3318', 'b1343', 'b3179', 'b3185', 'b3294', 'b3871', 'b1089', 'b3780', 'b3162', 'b3305', 'b3308', 'b0797'}, 'elements.common': set()}, {'id': 'GO:0000028', 'desc': 20, 'common.n': 0, 'target.n': 20, 'p-value': 1.0, 'elements.target': {'b2608', 'b3306', 'b2566', 'b3307', 'b3180', 'b2609', 'b3065', 'b0051', 'b3165', 'b3296', 'b3311', 'b4161', 'b0911', 'b3303', 'b3170', 'b0023', 'b3341', 'b3316', 'b3314

In [52]:
# results est une liste de dictionnaires
len(results)

2095

In [53]:
# afficher toutes les valeurs de la clé 'p-value' dans results
for r in results:
    print(r['p-value'])

1.0
1.0
1.0
1.0
1.0
0.08967886876720332
0.08967886876720332
0.0029421789415033928
1.0
4.141131881851834e-14
0.0021061210908088768
1.0
1.0
1.0
1.0
4.5499985890096184e-08
0.5255984680453407
1.0
1.0
1.0
1.0
0.17425996017167722
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.7538558806865269
0.7439516099528414
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
6.958071611107286e-07
1.0
0.9999950073846742
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.9236468529340979
0.9675956152782965
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
9.665515147982973e-05
0.08967886876720332
1.0
0.0021061210908088768
0.08967886876720332
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.2093458400889474
1.0
1.0
1.0
1.0
2.0356745311644886e-07
0.13145577666607156
1.0
1.0
0.13145577666607156
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.08967886876720332
0.0021061210908088768
1.0
1.0
0.00210612109080887

### 1.1.2.5 Affichage des résultats significatifs

In [54]:
significant_results = []
#count_significant = 0
# PRINT SIGNIFICANT RESULTS
results.sort(key=lambda an_item: an_item['p-value'])
for r in results:
    if r['p-value'] <= param.alpha: 
      significant_results.append((r['p-value']))
      #count_significant += 1
      #break
    # OUTPUT
    print("{}\t{}\t{}/{}\t{}\t{}".format( r['id'], r['p-value'], r['common.n'], r['target.n'], r['desc'], ', '.join(r['elements.common'])))

GO:0006865	0.0	24/28	28	b2923, b3268, b0861, b1296, b2307, b2128, b3270, b3269, b2130, b0576, b0810, b1336, b3089, b0654, b2014, b3116, b1492, b1605, b0486, b2131, b3795, b2129, b0653, b1453
GO:0008652	0.0	18/19	19	b3766, b3773, b0033, b1693, b0754, b0908, b3390, b2601, b0388, b3281, b3769, b0674, b0031, b1704, b2414, b2329, b3389, b4388
GO:0055052	0.0	37/150	150	b0863, b2308, b3268, b1918, b0861, b2310, b3455, b1917, b2307, b0199, b2128, b3270, b2306, b0652, b3269, b2130, b0198, b0862, b0810, b2309, b3458, b0860, b0197, b0654, b3460, b3271, b3456, b3457, b2131, b1920, b2129, b0811, b0653, b0655, b0809, b0864, b3454
GO:0009073	1.1102230246251565e-16	12/12	12	b1261, b1693, b0754, b0908, b0388, b2601, b3281, b3390, b1704, b1260, b2329, b3389
GO:0009086	2.1871393585115584e-14	11/12	12	b3828, b0529, b3940, b4019, b3008, b0261, b3938, b3829, b1622, b3941, b3939
GO:0009097	3.4861002973229915e-14	12/15	15	b3770, b3671, b0077, b0078, b3769, b3774, b4243, b3670, b3433, b3772, b3771, b4488
GO:00

Exemple d’utilisation avec set.01.txt :
./scripts/blastsets.json.py -q query.sets/set.01.txt -t reference.sets/uniprot.keywords.sets.json  | column -ts $'\t'
 ### POur avoir le nombre de res significatif
./scripts/blastsets.json.py --adjust -q query.sets/set.01.txt -t reference.sets/uniprot.keywords.sets.json | wc -l

In [55]:
print(f"Nombre de résultats significatifs : {len(significant_results)}") # dois avoir 41

Nombre de résultats significatifs : 153


#### FDR avec p valeur ajustée
P+m / i et si p_adjusterd > 1 alpha = 1

In [56]:
results[0]['p-value']

0.0

In [57]:
significant_results = []
#count_significant = 0
# PRINT SIGNIFICANT RESULTS
results.sort(key=lambda an_item: an_item['p-value'])
for r in results:
  if r['p-value'] < param.alpha: 
    significant_results.append((r['p-value']))
    print("{}\t{}\t{}/{}\t{}\t{}".format( r['id'], r['p-value'], r['common.n'], r['target.n'], r['desc'], ', '.join(r['elements.common'])))

  if param.adjust:
    p_ajusted = []
    for i, proba in enumerate(significant_results):
      p = (proba*len_sets) / (i+1)
      if p <= param.alpha :
        p_ajusted.append(p) 

GO:0006865	0.0	24/28	28	b2923, b3268, b0861, b1296, b2307, b2128, b3270, b3269, b2130, b0576, b0810, b1336, b3089, b0654, b2014, b3116, b1492, b1605, b0486, b2131, b3795, b2129, b0653, b1453
GO:0008652	0.0	18/19	19	b3766, b3773, b0033, b1693, b0754, b0908, b3390, b2601, b0388, b3281, b3769, b0674, b0031, b1704, b2414, b2329, b3389, b4388
GO:0055052	0.0	37/150	150	b0863, b2308, b3268, b1918, b0861, b2310, b3455, b1917, b2307, b0199, b2128, b3270, b2306, b0652, b3269, b2130, b0198, b0862, b0810, b2309, b3458, b0860, b0197, b0654, b3460, b3271, b3456, b3457, b2131, b1920, b2129, b0811, b0653, b0655, b0809, b0864, b3454
GO:0009073	1.1102230246251565e-16	12/12	12	b1261, b1693, b0754, b0908, b0388, b2601, b3281, b3390, b1704, b1260, b2329, b3389
GO:0009086	2.1871393585115584e-14	11/12	12	b3828, b0529, b3940, b4019, b3008, b0261, b3938, b3829, b1622, b3941, b3939
GO:0009097	3.4861002973229915e-14	12/15	15	b3770, b3671, b0077, b0078, b3769, b3774, b4243, b3670, b3433, b3772, b3771, b4488
GO:00

In [58]:
len(p_ajusted) # 35 ok !!!

109

# hypergeometrique - 1.1.2.4 Test statistique - 

## IMport avec parser 

In [77]:
# param = parser.parse_args(['-q', '/chemin/vers/votre/fichier/query.txt', '-t', '/chemin/vers/votre/fichier/ref.sets/uniprot.keywords.sets.json'])
param = parser.parse_args(['-q','../query/set1.txt',
                           '-t','../ref.sets/uniprot.keywords.sets.json',
                           '-m', 'hypergeometric',
                           '-c','-v','-l','10'])

# LOAD QUERY
text = param.query
query = set()
if isfile(text):
    with open(text) as f:
        content = ' '.join(f.read().split('\n')).split()
        query |= set(content)
else: # parse string
    query |= set(text.split())

if param.verbose:
  print(f'query set: {query}')

# LOAD REFERENCE SETS
sets = json.loads(open(param.sets).read()) # set est une liste de dictionnaires
if param.verbose:
    print('first target sets: ', sets[0:2])

query set: {'b2913', 'b0692', 'b3110', 'b2413', 'b1918', 'b3213', 'b4013', 'b2019', 'b2599', 'b3957', 'b0031', 'b1296', 'b1265', 'b2678', 'b3772', 'b2310', 'b0199', 'b3270', 'b2478', 'b3269', 'b4141', 'b0576', 'b2600', 'b2421', 'b0529', 'b0112', 'b3458', 'b3653', 'b0860', 'b1693', 'b1262', 'b2014', 'b0032', 'b1473', 'b3457', 'b0388', 'b1920', 'b0001', 'b0811', 'b3941', 'b0809', 'b3237', 'b2414', 'b2329', 'b3771', 'b2578', 'b2551', 'b2308', 'b4208', 'b4024', 'b2763', 'b3268', 'b4115', 'b3774', 'b2025', 'b0166', 'b1729', 'b3709', 'b3455', 'b3770', 'b2307', 'b3370', 'b0033', 'b2018', 'b4019', 'b2764', 'b0242', 'b0273', 'b0003', 'b0402', 'b0386', 'b3940', 'b4054', 'b0074', 'b0159', 'b0654', 'b3829', 'b3116', 'b1704', 'b3670', 'b3271', 'b3939', 'b1605', 'b3212', 'b1015', 'b2131', 'b0071', 'b3281', 'b4243', 'b2129', 'b2677', 'b4254', 'b3672', 'b3389', 'b2156', 'b4488', 'b2923', 'b3161', 'b0863', 'b3671', 'b2365', 'b2663', 'b0077', 'b0754', 'b0861', 'b4132', 'b1907', 'b2023', 'b1798', 'b1917'

In [78]:
sets

[{'id': '2Fe-2S',
  'desc': 26,
  'elements': ['b0286',
   'b0724',
   'b0775',
   'b0872',
   'b0873',
   'b1392',
   'b1654',
   'b1684',
   'b1802',
   'b1803',
   'b2236',
   'b2283',
   'b2285',
   'b2525',
   'b2530',
   'b2531',
   'b2538',
   'b2540',
   'b2868',
   'b2881',
   'b3337',
   'b3771',
   'b4063',
   'b4153',
   'b4178',
   'b4367']},
 {'id': '3D-structure',
  'desc': 1665,
  'elements': ['b0002',
   'b0004',
   'b0006',
   'b0008',
   'b0009',
   'b0010',
   'b0014',
   'b0015',
   'b0019',
   'b0023',
   'b0028',
   'b0029',
   'b0031',
   'b0032',
   'b0033',
   'b0038',
   'b0040',
   'b0046',
   'b0047',
   'b0048',
   'b0051',
   'b0052',
   'b0053',
   'b0054',
   'b0058',
   'b0059',
   'b0060',
   'b0061',
   'b0062',
   'b0064',
   'b0068',
   'b0073',
   'b0076',
   'b0078',
   'b0080',
   'b0082',
   'b0084',
   'b0085',
   'b0086',
   'b0088',
   'b0090',
   'b0091',
   'b0092',
   'b0093',
   'b0094',
   'b0095',
   'b0096',
   'b0097',
   'b0098',
  

### 1.1.2.4.1 Taille de la population

In [67]:
len_query = len(query)
len_sets = len(sets) # nombre d ensembles de reference
len_query, len_sets

(181, 334)

In [68]:
sets[0].keys()

dict_keys(['id', 'desc', 'elements'])

In [69]:
# j accede a la liste elements dans le premier dictionnaire de sets
sets[0]['elements']

['b0286',
 'b0724',
 'b0775',
 'b0872',
 'b0873',
 'b1392',
 'b1654',
 'b1684',
 'b1802',
 'b1803',
 'b2236',
 'b2283',
 'b2285',
 'b2525',
 'b2530',
 'b2531',
 'b2538',
 'b2540',
 'b2868',
 'b2881',
 'b3337',
 'b3771',
 'b4063',
 'b4153',
 'b4178',
 'b4367']

In [70]:
# et pour avoir la longeuer de cette liste :
len(sets[0]['elements']), len(sets[1]['elements']) #len de sets = 

(26, 1665)

In [71]:
# COMPUTE POPULATION SIZE, deduire la taille de la population dans ref set
population = set() # faire en sorte que population soit un ensemble
for s in sets:
  valeur_cle3 = s.get('elements')
  population.update(valeur_cle3)

# Maintenant, "valeurs_uniques" contient toutes les valeurs uniques de la troisième clé ('cle3')
n_gene_total = len(population)

# Affichez le nombre de valeurs uniques d elements dans la population
print(n_gene_total) # 4400

4400


### 1.1.2.4.2 Comparaison des ensembles cibles à l’ensemble requête

In [72]:
len(sets[0]['elements'])

26

In [73]:
p = len(sets[5]['elements']) / n_gene_total #probabilite de succes a chaque essai
p

0.00045454545454545455

In [88]:
# param = parser.parse_args(['-q', '/chemin/vers/votre/fichier/query.txt', '-t', '/chemin/vers/votre/fichier/ref.sets/uniprot.keywords.sets.json'])
param = parser.parse_args(['-q','../query/set1.txt',
                           '-t','../ref.sets/uniprot.keywords.sets.json',
                           '-m', 'hypergeometric',
                           '-c','-v','-l','10'])
# hypergeometric coverage chi2

In [89]:
# EVALUATE SETS
results = []
query_size = len(query)
for s in sets:
    elements = set(s['elements']) # liste d elements unique dans elements
    # define k as  observed.size - sum(observed.shape) + observed.ndim - 1
    common_elements = elements.intersection(query) #k, nombre de succes dans la population = commun entre query et elements
    k = len(common_elements) # nombre de succes dans la population = commun entre query et elements
    if param.measure=='binomial': # par défaut binom.cdf(<=success, attempts, proba). il nous faut calculer p(X>=x)
        p = 1 - binom.cdf(k - 1, len(elements), query_size / n_gene_total)
    if param.measure == 'coverage':
        p = len(sets[5]['elements']) / n_gene_total
    if param.measure == 'hypergeometric':
        # hypergeom.cdf(k, N, n, M, loc=0)
        p = 1 - hypergeom.cdf(k - 1, n_gene_total, len(elements), query_size)
    if param.measure == 'chi2': #cdf(x, df, loc=0, scale=1)
        # inpendant test of chi2 with k ddl
        p = 1 - chi2.cdf()

    r = { 'id': s['id'], 
         'desc': s['desc'], 
         'common.n':len(common_elements), 
         'target.n': len(elements), 
         'p-value': p, 
         'elements.target': elements, 
         'elements.common': common_elements }
    results.append( r )

if param.verbose:
  print(results)

[{'id': '2Fe-2S', 'desc': 26, 'common.n': 1, 'target.n': 26, 'p-value': 0.66557679249526, 'elements.target': {'b4367', 'b4153', 'b2868', 'b0872', 'b1392', 'b2530', 'b2538', 'b2236', 'b1803', 'b2881', 'b0775', 'b1684', 'b2531', 'b1654', 'b3337', 'b2525', 'b0873', 'b0724', 'b2283', 'b4178', 'b2285', 'b0286', 'b2540', 'b4063', 'b3771', 'b1802'}, 'elements.common': {'b3771'}}, {'id': '3D-structure', 'desc': 1665, 'common.n': 70, 'target.n': 1665, 'p-value': 0.43511012605148547, 'elements.target': {'b2287', 'b0407', 'b1211', 'b2411', 'b4101', 'b3320', 'b3772', 'b1652', 'b2918', 'b2373', 'b2894', 'b1745', 'b2562', 'b4035', 'b4025', 'b4375', 'b2288', 'b4090', 'b0758', 'b1159', 'b0345', 'b2441', 'b1987', 'b3869', 'b0778', 'b0881', 'b2371', 'b1045', 'b3620', 'b0565', 'b0480', 'b1625', 'b4501', 'b3400', 'b4034', 'b2723', 'b3190', 'b2385', 'b0897', 'b1635', 'b3786', 'b2595', 'b4311', 'b0123', 'b2905', 'b1100', 'b1636', 'b0096', 'b3651', 'b3619', 'b2942', 'b1390', 'b2910', 'b3707', 'b0907', 'b2528

In [90]:
results[3]['p-value']

0.8430089980614962

In [ ]:
# results est une liste de dictionnaires
len(results)

2095

In [ ]:
# afficher toutes les valeurs de la clé 'p-value' dans results
for r in results:
    print(r['p-value'])

1.0
1.0
1.0
1.0
1.0
0.08967886876720332
0.08967886876720332
0.0029421789415033928
1.0
4.141131881851834e-14
0.0021061210908088768
1.0
1.0
1.0
1.0
4.5499985890096184e-08
0.5255984680453407
1.0
1.0
1.0
1.0
0.17425996017167722
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.7538558806865269
0.7439516099528414
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
6.958071611107286e-07
1.0
0.9999950073846742
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.9236468529340979
0.9675956152782965
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
9.665515147982973e-05
0.08967886876720332
1.0
0.0021061210908088768
0.08967886876720332
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.2093458400889474
1.0
1.0
1.0
1.0
2.0356745311644886e-07
0.13145577666607156
1.0
1.0
0.13145577666607156
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.08967886876720332
0.0021061210908088768
1.0
1.0
0.00210612109080887

### 1.1.2.5 Affichage des résultats significatifs

In [ ]:
significant_results = []
#count_significant = 0
# PRINT SIGNIFICANT RESULTS
results.sort(key=lambda an_item: an_item['p-value'])
for r in results:
    if r['p-value'] <= param.alpha: 
      significant_results.append((r['p-value']))
      #count_significant += 1
      #break
    # OUTPUT
    print("{}\t{}\t{}/{}\t{}\t{}".format( r['id'], r['p-value'], r['common.n'], r['target.n'], r['desc'], ', '.join(r['elements.common'])))

GO:0006865	0.0	24/28	28	b2923, b3268, b0861, b1296, b2307, b2128, b3270, b3269, b2130, b0576, b0810, b1336, b3089, b0654, b2014, b3116, b1492, b1605, b0486, b2131, b3795, b2129, b0653, b1453
GO:0008652	0.0	18/19	19	b3766, b3773, b0033, b1693, b0754, b0908, b3390, b2601, b0388, b3281, b3769, b0674, b0031, b1704, b2414, b2329, b3389, b4388
GO:0055052	0.0	37/150	150	b0863, b2308, b3268, b1918, b0861, b2310, b3455, b1917, b2307, b0199, b2128, b3270, b2306, b0652, b3269, b2130, b0198, b0862, b0810, b2309, b3458, b0860, b0197, b0654, b3460, b3271, b3456, b3457, b2131, b1920, b2129, b0811, b0653, b0655, b0809, b0864, b3454
GO:0009073	1.1102230246251565e-16	12/12	12	b1261, b1693, b0754, b0908, b0388, b2601, b3281, b3390, b1704, b1260, b2329, b3389
GO:0009086	2.1871393585115584e-14	11/12	12	b3828, b0529, b3940, b4019, b3008, b0261, b3938, b3829, b1622, b3941, b3939
GO:0009097	3.4861002973229915e-14	12/15	15	b3770, b3671, b0077, b0078, b3769, b3774, b4243, b3670, b3433, b3772, b3771, b4488
GO:00

Exemple d’utilisation avec set.01.txt :
./scripts/blastsets.json.py -q query.sets/set.01.txt -t reference.sets/uniprot.keywords.sets.json  | column -ts $'\t'
 ### POur avoir le nombre de res significatif
./scripts/blastsets.json.py --adjust -q query.sets/set.01.txt -t reference.sets/uniprot.keywords.sets.json | wc -l

In [ ]:
print(f"Nombre de résultats significatifs : {len(significant_results)}") # dois avoir 41

Nombre de résultats significatifs : 153


#### FDR avec p valeur ajustée
P+m / i et si p_adjusterd > 1 alpha = 1

In [ ]:
results[0]['p-value']

0.0

In [ ]:
significant_results = []
#count_significant = 0
# PRINT SIGNIFICANT RESULTS
results.sort(key=lambda an_item: an_item['p-value'])
for r in results:
  if r['p-value'] < param.alpha: 
    significant_results.append((r['p-value']))
    print("{}\t{}\t{}/{}\t{}\t{}".format( r['id'], r['p-value'], r['common.n'], r['target.n'], r['desc'], ', '.join(r['elements.common'])))

  if param.adjust:
    p_ajusted = []
    for i, proba in enumerate(significant_results):
      p = (proba*len_sets) / (i+1)
      if p <= param.alpha :
        p_ajusted.append(p) 

GO:0006865	0.0	24/28	28	b2923, b3268, b0861, b1296, b2307, b2128, b3270, b3269, b2130, b0576, b0810, b1336, b3089, b0654, b2014, b3116, b1492, b1605, b0486, b2131, b3795, b2129, b0653, b1453
GO:0008652	0.0	18/19	19	b3766, b3773, b0033, b1693, b0754, b0908, b3390, b2601, b0388, b3281, b3769, b0674, b0031, b1704, b2414, b2329, b3389, b4388
GO:0055052	0.0	37/150	150	b0863, b2308, b3268, b1918, b0861, b2310, b3455, b1917, b2307, b0199, b2128, b3270, b2306, b0652, b3269, b2130, b0198, b0862, b0810, b2309, b3458, b0860, b0197, b0654, b3460, b3271, b3456, b3457, b2131, b1920, b2129, b0811, b0653, b0655, b0809, b0864, b3454
GO:0009073	1.1102230246251565e-16	12/12	12	b1261, b1693, b0754, b0908, b0388, b2601, b3281, b3390, b1704, b1260, b2329, b3389
GO:0009086	2.1871393585115584e-14	11/12	12	b3828, b0529, b3940, b4019, b3008, b0261, b3938, b3829, b1622, b3941, b3939
GO:0009097	3.4861002973229915e-14	12/15	15	b3770, b3671, b0077, b0078, b3769, b3774, b4243, b3670, b3433, b3772, b3771, b4488
GO:00

In [ ]:
len(p_ajusted) # 35 ok !!!

109